In [254]:
Text = "Neural networks have become an essential tool in the field of machine learning and data analysis. \
        Their ability to approximate complex nonlinear functions makes them suitable for a wide range of tasks, \
        including image classification, natural language processing, and time series prediction. \
        Recent advances in deep learning architectures, such as convolutional neural networks and recurrent neural networks, \
        have significantly improved performance across various domains. \
        In addition, optimization algorithms like Adam and RMSProp have facilitated faster convergence during training. \
        Despite these advancements, challenges remain in the areas of generalization, interpretability, and robustness \
        to adversarial inputs. \
        Ongoing research focuses on developing more efficient training methods, understanding theoretical foundations, \
        and integrating neural networks with symbolic reasoning to enhance their capabilities. \
        The synergy between algorithmic improvements, hardware acceleration, and large-scale datasets continues \
        to drive the rapid evolution of neural network applications across science and industry."

#reference: https://www.researchgate.net/publication/342626761_News_Aggregator_and_Efficient_Summarization_System
#reference: https://arxiv.org/abs/1704.01851 : co occurence in a window, undirected graph

In [255]:
import nltk
import string

import numpy as np
import math

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import word_tokenize, pos_tag

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [256]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default

def preprocess_text(text):
    # Tokenize the text
    tokenized_text = word_tokenize(text)
    tagged = pos_tag(tokenized_text)
    # Remove punctuation, convert to lowercase, lemmatize
    tokens = [lemmatizer.lemmatize(word.lower(), get_wordnet_pos(pos)) for word,pos in tagged if word not in string.punctuation and word not in stop_words]
    tokens_2 = [lemmatizer.lemmatize(word.lower(), get_wordnet_pos(pos)) for word,pos in tagged]

    return tokens,  tokens_2

preprocessed_text, tokens=preprocess_text(Text)

In [257]:
print(preprocessed_text)

['neural', 'network', 'become', 'essential', 'tool', 'field', 'machine', 'learning', 'data', 'analysis', 'their', 'ability', 'approximate', 'complex', 'nonlinear', 'function', 'make', 'suitable', 'wide', 'range', 'task', 'include', 'image', 'classification', 'natural', 'language', 'processing', 'time', 'series', 'prediction', 'recent', 'advance', 'deep', 'learning', 'architecture', 'convolutional', 'neural', 'network', 'recurrent', 'neural', 'network', 'significantly', 'improve', 'performance', 'across', 'various', 'domain', 'in', 'addition', 'optimization', 'algorithm', 'like', 'adam', 'rmsprop', 'facilitate', 'faster', 'convergence', 'training', 'despite', 'advancement', 'challenge', 'remain', 'area', 'generalization', 'interpretability', 'robustness', 'adversarial', 'input', 'ongoing', 'research', 'focus', 'develop', 'efficient', 'training', 'method', 'understand', 'theoretical', 'foundation', 'integrate', 'neural', 'network', 'symbolic', 'reason', 'enhance', 'capability', 'the', 's

In [258]:
vocabulary = list(set(preprocessed_text))
print (vocabulary)

['time', 'advancement', 'symbolic', 'tool', 'reason', 'theoretical', 'rmsprop', 'drive', 'addition', 'optimization', 'convergence', 'complex', 'enhance', 'network', 'generalization', 'interpretability', 'deep', 'efficient', 'significantly', 'ongoing', 'evolution', 'synergy', 'in', 'advance', 'range', 'recent', 'ability', 'research', 'adversarial', 'performance', 'convolutional', 'learning', 'make', 'include', 'improvement', 'algorithmic', 'nonlinear', 'across', 'training', 'application', 'integrate', 'function', 'language', 'the', 'neural', 'capability', 'processing', 'suitable', 'prediction', 'algorithm', 'natural', 'focus', 'acceleration', 'domain', 'various', 'analysis', 'field', 'continue', 'approximate', 'hardware', 'become', 'task', 'like', 'input', 'foundation', 'industry', 'understand', 'their', 'recurrent', 'remain', 'rapid', 'robustness', 'area', 'faster', 'method', 'architecture', 'classification', 'wide', 'datasets', 'challenge', 'develop', 'large-scale', 'machine', 'data',

In [259]:
import numpy as np
from collections import defaultdict
import math

vocab_index = {word: idx for idx, word in enumerate(vocabulary)}
vocab_len = len(vocabulary)
weighted_edge = np.zeros((vocab_len, vocab_len), dtype=np.float32)
score = np.ones(vocab_len, dtype=np.float32)/vocab_len
window_dim = 3
cooccurrences = set()

# moving window
for start in range(len(preprocessed_text) - window_dim + 1):
    window = preprocessed_text[start:start + window_dim]

    # linking couple of word in the same window
    for i in range(len(window)):
        for j in range(i + 1, len(window)):
            word_i, word_j = window[i], window[j]
            if word_i in vocab_index and word_j in vocab_index:
                idx_i, idx_j = vocab_index[word_i], vocab_index[word_j]
                index_pair = tuple(sorted((start + i, start + j)))

                if index_pair not in cooccurrences:
                    distance = abs(i - j)
                    if distance > 0:
                        weight = 1.0 / distance
                        weighted_edge[idx_i][idx_j] += weight
                        weighted_edge[idx_j][idx_i] += weight  # symmmetry, not oriented graph
                        cooccurrences.add(index_pair)

print(np.sum(score))

0.9999999


In [260]:
#pip install gensim

In [261]:
'''
# alternative for sentence ranking in order to have a summary by considering the top ranked sentences.
# the procedure in similar excpet for using the similarity matrix computed between the embedded sentences

import numpy as np
from gensim.models import Word2Vec

vocab_len = len(vocabulary)
score = np.zeros((vocab_len),dtype=np.float32)
# Train a Word2Vec model on your preprocessed text
# You can adjust the parameters like vector_size, window, min_count, etc.
model = Word2Vec([preprocessed_text], vector_size=5, window=3, min_count=1, workers=4)

#weighted_edge = weighted_edge
# Build the similarity matrix
weighted_edge = np.zeros((vocab_len, vocab_len), dtype=np.float32)

for i in range(vocab_len):
    score[i]=1
    for j in range(vocab_len):
        if i == j:
            weighted_edge[i][j] = 1.0  # A word is perfectly similar to itself
        else:
            word1 = vocabulary[i]
            word2 = vocabulary[j]
            # Check if both words are in the vocabulary of the Word2Vec model
            if word1 in model.wv and word2 in model.wv:
                weighted_edge[i][j] = model.wv.similarity(word1, word2)
            else:
                # Handle cases where one or both words are not in the model's vocabulary
                # You might set similarity to 0 or handle it differently based on your needs
                weighted_edge[i][j] = 0.0

#print("\nSimilarity Matrix (using Gensim Word2Vec):")

for i in range(vocab_len):
  for j in range(vocab_len):
    weighted_edge[i][j]=(weighted_edge[i][j]+1)/2
'''

'\n# alternative for sentence ranking in order to have a summary by considering the top ranked sentences.\n# the procedure in similar excpet for using the similarity matrix computed between the embedded sentences\n\nimport numpy as np\nfrom gensim.models import Word2Vec\n\nvocab_len = len(vocabulary)\nscore = np.zeros((vocab_len),dtype=np.float32)\n# Train a Word2Vec model on your preprocessed text\n# You can adjust the parameters like vector_size, window, min_count, etc.\nmodel = Word2Vec([preprocessed_text], vector_size=5, window=3, min_count=1, workers=4)\n\n#weighted_edge = weighted_edge\n# Build the similarity matrix\nweighted_edge = np.zeros((vocab_len, vocab_len), dtype=np.float32)\n\nfor i in range(vocab_len):\n    score[i]=1\n    for j in range(vocab_len):\n        if i == j:\n            weighted_edge[i][j] = 1.0  # A word is perfectly similar to itself\n        else:\n            word1 = vocabulary[i]\n            word2 = vocabulary[j]\n            # Check if both words are 

In [262]:
row_sum = np.zeros((vocab_len),dtype=np.float32)

#summing the entries on each rows

for i in range(0,vocab_len):
    for j in range(0,vocab_len):
        row_sum[i]+=weighted_edge[i][j]


In [263]:
MAX_ITERATIONS = 50
d=0.75
threshold = 0.0001 #convergence threshold

iter=0
prev_score = np.zeros(len(score))

while iter < MAX_ITERATIONS and np.sum(np.fabs(prev_score-score)) >= threshold:
    prev_score = np.copy(score)

    for i in range(0,vocab_len):

        summation = 0
        for j in range(0,vocab_len):
            if weighted_edge[i][j] != 0:
                summation += (weighted_edge[i][j]/(row_sum[j]))*score[j]

        score[i] = (1-d)/vocab_len + d*(summation)
    iter+=1

print("Converging at iteration "+str(iter))
#print(np.sum(score))

Converging at iteration 10
1.0000993


In [264]:
for i in range(0,vocab_len):
    #if score[i]>1:
    print("Score of "+vocabulary[i]+": "+str(score[i]))

Score of time: 0.010802251
Score of advancement: 0.010363465
Score of symbolic: 0.008753399
Score of tool: 0.00984636
Score of reason: 0.009539557
Score of theoretical: 0.009892883
Score of rmsprop: 0.0106993485
Score of drive: 0.010133029
Score of addition: 0.010669536
Score of optimization: 0.010745045
Score of convergence: 0.010142585
Score of complex: 0.010807938
Score of enhance: 0.010133946
Score of network: 0.030743318
Score of generalization: 0.010812339
Score of interpretability: 0.010830241
Score of deep: 0.009869584
Score of efficient: 0.010117379
Score of significantly: 0.008565231
Score of ongoing: 0.010774778
Score of evolution: 0.008750461
Score of synergy: 0.010721177
Score of in: 0.010539042
Score of advance: 0.010224765
Score of range: 0.010866893
Score of recent: 0.010517259
Score of ability: 0.01067855
Score of research: 0.010705476
Score of adversarial: 0.010829779
Score of performance: 0.009485404
Score of convolutional: 0.008428958
Score of learning: 0.017238919


In [265]:
#here
groups = []

group = " "

print(tokens)

for word in tokens:
    #in this for loop adjacent words are grouped if there is no punctuation or stopword between them
    if word in stop_words or word in string.punctuation:
        if group!= " ":
            groups.append(str(group).strip().split())
        group = " "
    elif word not in stop_words:
        group+=str(word)
        group+=" "

print("Grouped words: \n")
print(groups)

['neural', 'network', 'have', 'become', 'an', 'essential', 'tool', 'in', 'the', 'field', 'of', 'machine', 'learning', 'and', 'data', 'analysis', '.', 'their', 'ability', 'to', 'approximate', 'complex', 'nonlinear', 'function', 'make', 'them', 'suitable', 'for', 'a', 'wide', 'range', 'of', 'task', ',', 'include', 'image', 'classification', ',', 'natural', 'language', 'processing', ',', 'and', 'time', 'series', 'prediction', '.', 'recent', 'advance', 'in', 'deep', 'learning', 'architecture', ',', 'such', 'a', 'convolutional', 'neural', 'network', 'and', 'recurrent', 'neural', 'network', ',', 'have', 'significantly', 'improve', 'performance', 'across', 'various', 'domain', '.', 'in', 'addition', ',', 'optimization', 'algorithm', 'like', 'adam', 'and', 'rmsprop', 'have', 'facilitate', 'faster', 'convergence', 'during', 'training', '.', 'despite', 'these', 'advancement', ',', 'challenge', 'remain', 'in', 'the', 'area', 'of', 'generalization', ',', 'interpretability', ',', 'and', 'robustness

In [266]:
unique_groups = []

#check to be sure to consider only one time a group formed if two or more are formed
for group in groups:
    if group not in unique_groups:
        unique_groups.append(group)

print("Unique groups: \n")
print(unique_groups)

Unique groups: 

[['neural', 'network'], ['become'], ['essential', 'tool'], ['field'], ['machine', 'learning'], ['data', 'analysis'], ['ability'], ['approximate', 'complex', 'nonlinear', 'function', 'make'], ['suitable'], ['wide', 'range'], ['task'], ['include', 'image', 'classification'], ['natural', 'language', 'processing'], ['time', 'series', 'prediction'], ['recent', 'advance'], ['deep', 'learning', 'architecture'], ['convolutional', 'neural', 'network'], ['recurrent', 'neural', 'network'], ['significantly', 'improve', 'performance', 'across', 'various', 'domain'], ['addition'], ['optimization', 'algorithm', 'like', 'adam'], ['rmsprop'], ['facilitate', 'faster', 'convergence'], ['training'], ['despite'], ['advancement'], ['challenge', 'remain'], ['area'], ['generalization'], ['interpretability'], ['robustness'], ['adversarial', 'input'], ['ongoing', 'research', 'focus'], ['develop'], ['efficient', 'training', 'method'], ['understand', 'theoretical', 'foundation'], ['integrate', 'n

In [267]:
group_scores = []
strings = []

for group in unique_groups:
    group_score=0
    string = ''
    for word in group:
        string += str(word)
        string += " "
        group_score+=score[vocabulary.index(word)]   #for each group of words the is the sum of the scores of each word divided by the number of word that compose the group
    group_scores.append(group_score/len(group))
    strings.append(string)

i=0
for keyword in strings:
    print ("Keyword: '"+str(keyword)+"', Score: "+str(group_scores[i]))
    i+=1

Keyword: 'neural network ', Score: 0.029652921
Keyword: 'become ', Score: 0.008676691
Keyword: 'essential tool ', Score: 0.009612592
Keyword: 'field ', Score: 0.009884663
Keyword: 'machine learning ', Score: 0.013530603
Keyword: 'data analysis ', Score: 0.010123167
Keyword: 'ability ', Score: 0.01067855
Keyword: 'approximate complex nonlinear function make ', Score: 0.010822355
Keyword: 'suitable ', Score: 0.01086294
Keyword: 'wide range ', Score: 0.010866211
Keyword: 'task ', Score: 0.010867142
Keyword: 'include image classification ', Score: 0.0108646415
Keyword: 'natural language processing ', Score: 0.010845129
Keyword: 'time series prediction ', Score: 0.0107392585
Keyword: 'recent advance ', Score: 0.010371013
Keyword: 'deep learning architecture ', Score: 0.012038366
Keyword: 'convolutional neural network ', Score: 0.022578267
Keyword: 'recurrent neural network ', Score: 0.022262901
Keyword: 'significantly improve performance across various domain ', Score: 0.010795716
Keyword: 

In [268]:
sorted_index = np.flip(np.argsort(group_scores),0) #argsort ascending order -> flip descending

keywords_num = 3

print("Keywords:\n")

for i in range(0,keywords_num):
    print(str(strings[sorted_index[i]])+"\n")

Keywords:

neural network 

integrate neural network 

convolutional neural network 

